# Implement Tarrataca's quantum production system with the 3-puzzle problem
**Computer Laboratory II (417526) – Quantum AI – Practical 3**

Install once (Linux terminal):
```
pip install qiskit qiskit-aer matplotlib pylatexenc
```
Runs on the local Aer simulator – no IBM account or API token needed (the lab manual's `execute` / `IBMQ` / `qiskit.aqua` / `qiskit.ignis` code was removed from current Qiskit, so this notebook uses the current API).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

import qiskit, qiskit_aer
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

sim = AerSimulator()
print("qiskit", qiskit.__version__, "| qiskit-aer", qiskit_aer.__version__)

def run(qc, shots=1024, backend=sim, **kw):
    # Transpile for the backend, run, and return the counts dictionary.
    t = transpile(qc, backend, **kw)
    return backend.run(t, shots=shots).result().get_counts()

## Practical 3 – Tarrataca's Quantum Production System with the 3-puzzle
**Theory.** A *production system* has (a) a **state**, (b) **production rules** that change a state, and (c) a **goal test**. Tarrataca's quantum production system represents the **sequence of rules** to apply in a quantum register. A uniform superposition holds *every* rule sequence at once, and **Grover's amplitude amplification** boosts the sequences that lead from the initial state to the goal.

**The 3-puzzle** is the 2×2 sliding puzzle: 3 tiles + 1 blank.
```
positions:  0 1        goal:  1 2
            2 3               3 _
```
The blank can only move along the ring 0→1→3→2→0, so every state has exactly two production rules: **0 = blank moves clockwise, 1 = blank moves anticlockwise**. Therefore one qubit encodes one move, and *L* qubits encode a sequence of *L* moves.

**Steps**
1. Encode rule sequences in L qubits and put them in superposition (Hadamards)
2. **Oracle**: flip the phase of every sequence that transforms the initial state into the goal
3. **Diffuser** (inversion about the mean) – repeat 2–3 about (π/4)·√(N/M) times
4. Measure the qubits → a valid solution sequence

> The manual's code for this experiment relies on `qiskit.aqua` (removed) and does not run. This is a working implementation of the same idea.

In [ ]:
import itertools

RING = [0, 1, 3, 2]          # board positions in the order the blank travels
GOAL = (1, 2, 3, 0)          # 0 = blank

def apply_move(state, bit):
    # production rule: bit 0 -> blank moves clockwise, bit 1 -> anticlockwise
    s = list(state)
    b = s.index(0)
    k = RING.index(b)
    nb = RING[(k + 1) % 4] if bit == 0 else RING[(k - 1) % 4]
    s[b], s[nb] = s[nb], s[b]
    return tuple(s)

def run_sequence(state, bits):
    for b in bits:
        state = apply_move(state, b)
    return state

def show(state):
    t = ['_' if v == 0 else str(v) for v in state]
    return f"{t[0]} {t[1]} / {t[2]} {t[3]}"

# Initial state = goal scrambled by a few moves
INITIAL = run_sequence(GOAL, [0, 0, 1])
L = 5                                         # number of moves (= number of qubits)

N = 2 ** L
solutions = [b for b in itertools.product([0, 1], repeat=L) if run_sequence(INITIAL, b) == GOAL]
M = len(solutions)
print("Initial :", show(INITIAL), "| Goal :", show(GOAL))
print(f"Search space N = {N} sequences, solutions M = {M}")
print("Some solutions (move k = qubit k):", solutions[:4], "...")

In [ ]:
def mcz(qc, qubits):
    # multi-controlled Z on the given qubits
    if len(qubits) == 1:
        qc.z(qubits[0])
    else:
        qc.h(qubits[-1]); qc.mcx(qubits[:-1], qubits[-1]); qc.h(qubits[-1])

def oracle(L, sols):
    # Phase-flips |bits> for every solution sequence (qubit k holds move k)
    qc = QuantumCircuit(L, name='Oracle')
    for sol in sols:
        zeros = [k for k, b in enumerate(sol) if b == 0]
        for k in zeros: qc.x(k)
        mcz(qc, list(range(L)))
        for k in zeros: qc.x(k)
    return qc

def diffuser(L):
    qc = QuantumCircuit(L, name='Diffuser')
    qc.h(range(L)); qc.x(range(L))
    mcz(qc, list(range(L)))
    qc.x(range(L)); qc.h(range(L))
    return qc

iterations = max(1, int(np.floor(np.pi / (4 * np.arcsin(np.sqrt(M / N))))))
print("Grover iterations:", iterations)

grover = QuantumCircuit(L, L)
grover.h(range(L))                                   # superposition of ALL rule sequences
for _ in range(iterations):
    grover.compose(oracle(L, solutions), inplace=True)
    grover.compose(diffuser(L), inplace=True)
grover.measure(range(L), range(L))

counts = run(grover, shots=2048)
ok = sum(v for k, v in counts.items() if tuple(int(ch) for ch in reversed(k)) in set(solutions))
print(f"Probability of measuring a valid solution: {ok/2048:.3f}   (random guess would give {M/N:.3f})")
plot_histogram(counts, figsize=(12, 4), title='Grover search over move sequences')

In [ ]:
best = max(counts, key=counts.get)
moves = [int(ch) for ch in reversed(best)]          # qubit k -> move k
print("Most frequent outcome:", best, "-> moves", moves, "(0=clockwise, 1=anticlockwise)\n")
state = INITIAL
print("start :", show(state))
for i, mv in enumerate(moves, 1):
    state = apply_move(state, mv)
    print(f"move {i} ({mv}):", show(state))
print("\nReached goal?", state == GOAL)

**Note:** the sequence found may contain back-and-forth moves (a move followed by its opposite cancels out). Because all L = 5 moves are used, sequences with such detours are valid solutions too. The shortest solution here has only 3 moves.